In [8]:
import polars as pl
import altair as alt
import pandas as pd  # for Datawrapper
import pyarrow
import geopandas as gpd
from shapely.geometry import Point

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [9]:
prisoner_df = pl.read_csv("../data/prisoner_dataset.csv")
facility_df = pl.read_csv("../data/latest_facility_counts.csv")

In [10]:
def cleaning(prisoner_df, facility_df):
    facility_df = facility_df.filter(pl.col("State") == "Texas").with_columns(
        pl.col("Name").str.split(by=" ").list.first(),
    )

    facility_df = facility_df.unique(subset=["Name"])

    facility_df = facility_df.with_columns(
        pl.col("Name").str.to_titlecase().alias("Current Facility")
    )

    cutoff_date = pl.lit("09/22/2025").str.to_date("%m/%d/%Y")  # From date published

    prisoner_df = prisoner_df.with_columns(
        pl.when(pl.col("Sentence (Years)").is_in(["Capital Life", "Life", "LWOP"]))
        .then(pl.lit("Yes"))
        .otherwise(pl.lit("No"))
        .alias("Life Sentence?")
    )

    prisoner_df = prisoner_df.with_columns(
        pl.col("TDCJ Offense").str.to_titlecase(),
        pl.col("Sentence Date").str.to_date("%m/%d/%Y"),
        pl.col("Parole Review Status").str.to_titlecase(),
    )

    prisoner_df = prisoner_df.with_columns(
        (cutoff_date - pl.col("Sentence Date")).dt.total_days().alias("Time Served"),
    )

    prisoner_df = prisoner_df.with_columns((pl.col("Time Served") / 365))

    mapping = {
        "Agg Sex Aslt Child": "Agg. Sexual Assault of a Child",
        "Agg Sexual Aslt Child": "Agg. Sexual Assault of a Child",
        "Agg Robery": "Agg.Robbery",
        "Agg Aslt W/Dwpn": "Agg. Assault with a Deadly Weapon",
        "Agg Aslt Dw": "Agg. Assault with a Deadly Weapon",
        "Burg Habit": "Burglary of a Habitation",
        "Dwi": "DWI",
        "Agg Sex Aslt": "Agg. Sexual Assault",
        "Agg Robbery": "Agg. Robbery",
        "Agg Sex Aslt Child U/14": "Agg. Sexual Assault of a Child",
        "Agg Aslt W/Dw": "Agg. Assault with a Deadly Weapon",
        "Agg Aslt W/Ddly Wpn": "Agg. Assault with a Deadly Weapon",
        "Burg Of Habit": "Burglary of a Habitation",
        "Agg Sexual Aslt": "Agg. Sexual Assault",
        "Agg Aslt Dwpn": "Agg. Assault with a Deadly Weapon",
        "Agg Aslt W/Deadly Wpn": "Agg. Assault with a Deadly Weapon",
        "Agg Sex Aslt Of Child": "Agg. Sexual Assault of a Child",
        "Agg Asslt W/Dwpn": "Agg. Assault with a Deadly Weapon",
        "Agg Sexual Aslt Of Child": "Agg. Sexual Assault of a Child",
        "Dwi 3rd Or More": "DWI",
        "Sex Aslt Child": "Sexual Assault of a Child",
        "Dwi 3rd/More": "DWI",
        "Indecency W/Child Sex Contact": "Indecency with a Child with Sexual Contact",
        "Indec W/Child Sex Contact": "Indecency with a Child with Sexual Contact",
        "Cap Murder": "Capital Murder",
        "Agg Sex Aslt-Child": "Agg. Sexual Assault of a Child",
        "Indecency W/Child": "Indecency with a Child",
        "Indec W/Child": "Indecency with a Child",
        "Dwi 3rd More": "DWI",
        "Indecency W/Child Contact": "Indecency with a Child with Sexual Contact",
        "Indecency W/Child Contact": "Indecency with a Child with Sexual Contact",
    }
    # Instead of doing some sort of string matching, because of the nature of the data, I decided to be more thorough and do it by hand.

    prisoner_df = prisoner_df.with_columns(
        pl.col("TDCJ Offense").replace(mapping).alias("TDCJ Offense")
    )

    over_fifty_df = prisoner_df.filter((pl.col("Age") >= 50))
    # 35,350 rows

    return prisoner_df, facility_df, over_fifty_df


prisoner_df, facility_df, over_fifty_df = cleaning(prisoner_df, facility_df)

In [11]:
def merging_and_small_datasets(prisoner_df, facility_df):
    combined_df = prisoner_df.join(facility_df, on="Current Facility")

    race_comparison_df = prisoner_df.filter(
        pl.col("Race").is_in(["W", "H", "B"]),
    )

    filtered = combined_df.filter(pl.col("Age") > 50)

    counts = filtered.group_by("Current Facility").len(name="count_over_50")

    facilities = combined_df.select("Current Facility", "Latitude", "Longitude").unique(
        subset=["Current Facility"]
    )

    prison_age_points = counts.join(facilities, on="Current Facility", how="left")

    return combined_df, race_comparison_df, prison_age_points


combined_df, race_comparison_df, prison_age_points = merging_and_small_datasets(
    prisoner_df, facility_df
)

In [12]:
def get_geospatial_data(prison_age_points):
    prison_age_point_pd = prison_age_points.to_pandas()

    gdf = gpd.GeoDataFrame(
        prison_age_point_pd,
        geometry=gpd.points_from_xy(
            prison_age_point_pd.Longitude, prison_age_point_pd.Latitude
        ),
        crs="EPSG:4326",  # WGS84 lat/lon
    )

    gdf = gdf[
        gdf["Current Facility"] != "Lindsey"
    ]  # Not showing up properly, causing map to be distorted

    texas = gpd.read_file("../data/Texas_State_Boundary.zip").to_crs("EPSG:4326")

    counties = gpd.read_file("../data/Texas_County_Boundaries_Detailed.zip").to_crs(
        "EPSG:4326"
    )

    county_temp_data = pd.read_csv("../data/avg_temp_prison_august_2023.csv")

    merged_county_df = counties.merge(
        county_temp_data, left_on="CNTY_NM", right_on="COUNTY", how="left"
    )

    return gdf, texas, merged_county_df


gdf, texas, merged_county_df = get_geospatial_data(prison_age_points)

In [13]:
# 10 Most Common Offenses in Texas Jails


def common_offenses(df):
    chart = alt.Chart(df)
    domain = ["No", "Yes"]
    range = ["#b0a1a1", "#44414f"]
    common_bar_chart = (
        chart.mark_bar()
        .encode(
            alt.Y("TDCJ Offense:N", title="Offense").sort("-x"),
            alt.X("count:Q", title="Number of Prisoners"),
            alt.Color("Life Sentence?:N").scale(domain=domain, range=range),
        )
        .transform_aggregate(
            count="count()", groupby=["TDCJ Offense", "Life Sentence?"]
        )
        .transform_window(
            rank="rank(count)", sort=[alt.SortField("count", order="descending")]
        )
        .transform_filter((alt.datum.rank <= 10))
        .properties(
            title=alt.TitleParams(
                text="Top 10 Most Common Charges for Inmates over 50",
                subtitle="Count as of August 2025",
            )
        )
    )

    footnote = (
        alt.Chart(
            {
                "values": [
                    {
                        "text": [
                            "Similar charges above 100 occurrences merged.",
                            "Source: Texas Department of Criminal Justice",
                        ]
                    }
                ]
            }
        )
        .mark_text(
            align="left",
            baseline="bottom",
            fontSize=10,
        )
        .encode(text=alt.Text("text:N"))
    )

    final_chart = alt.vconcat(common_bar_chart, footnote)

    final_chart.save("../images/charges.svg")


common_offenses(over_fifty_df)

In [14]:
# Age and average length of sentence


def age_sentence(df):
    chart = alt.Chart(df, title="Average Sentence Length and Time Served by Age")
    domain = ["Sentence (Years)", "Time Served"]
    range = ["#b0a1a1", "#44414f"]

    age_sentence_chart = (
        chart.transform_filter(
            (alt.datum["Sentence (Years)"] < 100), (alt.datum["Age"] < 80)
        )
        .transform_fold(["Sentence (Years)", "Time Served"], as_=["Legend", "value"])
        .transform_aggregate(mean_value="mean(value)", groupby=["Age", "Legend"])
        .mark_line()
        .encode(
            alt.X("Age:Q"),
            alt.Y("mean_value:Q", title="Years"),
            alt.Color("Legend:N").scale(domain=domain, range=range),
        )
    )

    footnote = (
        alt.Chart(
            {
                "values": [
                    {
                        "text": [
                            "Truncated people above 80, who are 200 of the 139,210 records.",
                            "Source: Texas Department of Criminal Justice",
                        ]
                    }
                ]
            }
        )
        .mark_text(
            align="left",
            baseline="bottom",
            fontSize=10,
        )
        .encode(text=alt.Text("text:N"))
    )

    final_chart = alt.vconcat(age_sentence_chart, footnote)

    final_chart.save("../images/age_line_graph.svg")


age_sentence(prisoner_df)

In [15]:
# Scatterchart of prisons by number of prisoners and staff


def prison_pop_scatter(df):
    chart = alt.Chart(df, title="Prisons by Inmate and Staff Count")
    prison_pop_scatter = (
        chart.mark_point()
        .transform_calculate(is_old="datum.Age > 50 ? 1 : 0")
        .transform_aggregate(
            count="count()",
            staff_confirmed="average(Staff.Confirmed)",
            old_count="sum(is_old)",
            groupby=["Current Facility"],
        )
        .transform_calculate(proportion_old="datum.old_count/datum.count > 0.25")
        .transform_filter("datum.proportion_old != null")
        .encode(
            alt.X("count:Q", title="Prisoners Count", axis=alt.Axis(tickCount=10)),
            alt.Y("staff_confirmed:Q", title="Staff Count"),
            alt.Shape(
                "proportion_old:N",
                title="Proportion of Inmates Age 50+",
                scale=alt.Scale(domain=[False, True], range=["square", "triangle-up"]),
                legend=alt.Legend(
                    orient="none",
                    legendX=400,
                    legendY=330,
                    labelExpr="datum.value ? '25% or higher' : 'Lower than 25%'",
                ),
            ),
            alt.Color(
                "proportion_old:N",
                title="Proportion of Inmates Age 50+",
                scale=alt.Scale(domain=[False, True], range=["#ffa600", "#665191"]),
                legend=alt.Legend(
                    labelExpr="datum.value ? '25% or higher' : 'Lower than 25%'"
                ),
            ),
        )
    )

    prison_pop_scatter = prison_pop_scatter.properties(width=600)

    footnote = (
        alt.Chart(
            {
                "values": [
                    {
                        "text": [
                            "Source: Texas Department of Criminal Justice, UCLA Law",
                            "Based on matching of TDCJ Records with UCLA Law Facilities",
                        ]
                    }
                ]
            }
        )
        .mark_text(
            align="left",
            baseline="bottom",
            fontSize=10,
        )
        .encode(text=alt.Text("text:N"))
    )

    final_chart = alt.vconcat(prison_pop_scatter, footnote)

    final_chart.save("../images/facility_scatterplot.svg")


prison_pop_scatter(combined_df)

In [16]:
# race on sentence time - line graph


def race_sentence(df):
    chart = alt.Chart(df, title="Average Time Served and Age by Race")
    domain = ["B", "H", "W"]
    range = ["#665191", "#bc5090", "#ffa600"]
    race_sentence_chart = (
        chart.mark_line()
        .encode(
            alt.X("Age:Q"),
            alt.Y("average(Time Served):Q", title="Time Served"),
            alt.Color(
                "Race:N",
                legend=alt.Legend(
                    labelExpr="datum.label == 'B' ? 'Black' : datum.label == 'H' ? 'Hispanic' : 'White'"
                ),
            ).scale(domain=domain, range=range),
        )
        .transform_filter((alt.datum["Time Served"] < 100), (alt.datum["Age"] < 80))
    )

    footnote = (
        alt.Chart(
            {
                "values": [
                    {
                        "text": [
                            "Truncated people above 80, who are 200 of the 139,210 records.",
                            "Source: Texas Department of Criminal Justice",
                        ]
                    }
                ]
            }
        )
        .mark_text(
            align="left",
            baseline="bottom",
            fontSize=10,
        )
        .encode(text=alt.Text("text:N"))
    )

    final_chart = alt.vconcat(race_sentence_chart, footnote)

    final_chart.save("../images/race_line_graph.svg")


race_sentence(race_comparison_df)

In [17]:
# race on sentence time - violin plot


def race_sentence_violin(df):
    df = df.filter(
        pl.col("Sentence (Years)").cast(pl.Float64, strict=False).is_not_null()
    )
    domain = ["B", "H", "W"]
    range = ["#665191", "#bc5090", "#ffa600"]
    chart = (
        alt.Chart(df, width=100, title="Density of Inmates Age 50+ By Race")
        .transform_density(
            "Age", as_=["Age", "density"], extent=[50, 90], groupby=["Race"]
        )
        .mark_area(orient="horizontal")
        .encode(
            alt.X("density:Q")
            .stack("center")
            .impute(None)
            .title(None)
            .axis(labels=False, grid=False, ticks=True),
            alt.Y("Age:Q"),
            alt.Color("Race:N", legend=None).scale(domain=domain, range=range),
            alt.Column("Race:N")
            .spacing(0)
            .header(
                titleOrient="bottom",
                labelOrient="bottom",
                labelPadding=0,
                labelExpr="datum.value == 'B' ? 'Black' : datum.value == 'H' ? 'Hispanic' : 'White'",
            ),
        )
    )

    footnote = (
        alt.Chart(
            {"values": [{"text": ["Source: Texas Department of Criminal Justice"]}]}
        )
        .mark_text(
            align="left",
            baseline="bottom",
            fontSize=10,
        )
        .encode(text=alt.Text("text:N"))
    )

    final_chart = alt.vconcat(chart, footnote, spacing=10).configure_view(stroke=None)

    final_chart.save("../images/race_violin_graph.svg")


race_sentence_violin(race_comparison_df)

In [18]:
# Parole review process status versus age histogram
def parole_review(df):
    domain = ["In Parole Review Process", "Not In Review Process"]
    range = ["#44414f", "#b0a1a1"]
    chart = alt.Chart(df, title="Parole Review Status by Age Groups")
    parole_histogram = chart.mark_bar().encode(
        alt.X("Age:Q", bin=True, title="Age Group"),
        alt.Y("count():Q", title="Number of Inmates"),
        alt.Color(
            "Parole Review Status:N",
            legend=alt.Legend(orient="none", legendX=160, legendY=50),
        ).scale(domain=domain, range=range),
    )

    footnote = (
        alt.Chart(
            {"values": [{"text": ["Source: Texas Department of Criminal Justice"]}]}
        )
        .mark_text(
            align="left",
            baseline="bottom",
            fontSize=10,
        )
        .encode(text=alt.Text("text:N"))
    )

    final_chart = alt.vconcat(parole_histogram, footnote)

    final_chart.save("../images/parole_review.svg")


parole_review(prisoner_df)

In [19]:
# AI: I was stuck on where to get started with simple geospatial mapping, so I asked ChatGPT to help me get started. See details in citations.md.

# use https://altair-viz.github.io/user_guide/marks/geoshape.html

In [20]:
# Choropleth


def choropleth_facility():
    basemap = alt.Chart(
        texas, title="Inmate Population Age 50+ and Average Prison Temperature"
    ).mark_geoshape(fill="lightgray", stroke="white", strokeWidth=0.5)

    bubbles = (
        alt.Chart(gdf)
        .mark_circle(stroke="black", opacity=0.4)
        .encode(
            longitude="Longitude:Q",
            latitude="Latitude:Q",
            size=alt.Size(
                "count_over_50:Q",
                scale=alt.Scale(range=[100, 800]),
                bin=alt.Bin(maxbins=4),
                title="Number of Prisoners Age 50+",
            ),
        )
        .properties(width=700, height=500)
    )

    background_counties = alt.Chart(merged_county_df).mark_geoshape(
        fill="white", stroke="gray", strokeWidth=0.5
    )

    counties_chart = (
        alt.Chart(merged_county_df)
        .mark_geoshape()
        .encode(
            fill=alt.Fill(
                "AVERAGE TEMPERATURE:Q",
                scale=alt.Scale(scheme="oranges"),
                title="Average Indoor Prison Temp.",
            )
        )
    )

    final_chart = basemap + background_counties + counties_chart + bubbles

    footnote = (
        alt.Chart(
            {
                "values": [
                    {
                        "text": [
                            "Average temperature chosen if multiple prisons in one county. Missing counties due to no prisons or no data.",
                            "Sources: Texas Department of Criminal Justice, UCLA Law, Texas Department of Transportation, FCC",
                        ]
                    }
                ]
            }
        )
        .mark_text(
            align="left",
            baseline="bottom",
            fontSize=10,
        )
        .encode(text=alt.Text("text:N"))
    )

    final_chart = alt.vconcat(final_chart, footnote)

    final_chart.save("../images/heatmap.svg")


choropleth_facility()